# CIFAR-10 Starter: Cache + Train/Test/Validation Split

This notebook downloads CIFAR-10 once, caches 10,000 images locally, and then splits the cached dataset into train/test/validation sets.

Environment note: this project is managed with `uv` (`uv sync` to install/update dependencies).

## 1) Imports and Paths
This section imports required libraries and defines all data/cache paths used in the notebook.

In [1]:
# Import libraries and define reusable project paths.
from pathlib import Path
import tarfile
import pickle

import numpy as np

CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"

ARCHIVE_PATH = RAW_DIR / "cifar-10-python.tar.gz"
EXTRACTED_DIR = RAW_DIR / "cifar-10-batches-py"
CACHE_PATH = CACHE_DIR / "cifar10_10000.npz"

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 2) Helper Functions
This section provides safe extraction, dataset download, and CIFAR batch loading utilities.

In [2]:
# Define reusable helpers for safe extraction and batch loading.
def _safe_extract(tar: tarfile.TarFile, path: Path) -> None:
    base = path.resolve()
    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise RuntimeError(f"Unsafe path in tar file: {member.name}")
    tar.extractall(path=path)


def download_and_extract_cifar10() -> None:
    import urllib.request

    if EXTRACTED_DIR.exists():
        print(f"Dataset already extracted at: {EXTRACTED_DIR}")
        return

    if not ARCHIVE_PATH.exists():
        print("Downloading CIFAR-10 archive...")
        urllib.request.urlretrieve(CIFAR10_URL, ARCHIVE_PATH)
        print(f"Saved archive to: {ARCHIVE_PATH}")
    else:
        print(f"Archive already exists at: {ARCHIVE_PATH}")

    print("Extracting archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        _safe_extract(tar, RAW_DIR)
    print(f"Extracted to: {EXTRACTED_DIR}")


def load_cifar_batch(batch_file: Path):
    with batch_file.open("rb") as f:
        batch = pickle.load(f, encoding="bytes")

    images = batch[b"data"]
    labels = np.array(batch[b"labels"], dtype=np.int64)

    images = images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images, labels


def split_dataset(images: np.ndarray, labels: np.ndarray, train_ratio: float, test_ratio: float, val_ratio: float, seed: int = 42):
    ratio_sum = train_ratio + test_ratio + val_ratio
    if not np.isclose(ratio_sum, 1.0):
        raise ValueError(f"train_ratio + test_ratio + val_ratio must be 1.0, got {ratio_sum:.4f}")

    n = len(images)
    rng = np.random.default_rng(seed)
    indices = np.arange(n)
    rng.shuffle(indices)

    train_end = int(n * train_ratio)
    test_end = train_end + int(n * test_ratio)

    train_idx = indices[:train_end]
    test_idx = indices[train_end:test_end]
    val_idx = indices[test_end:]

    return {
        "X_train": images[train_idx],
        "y_train": labels[train_idx],
        "X_test": images[test_idx],
        "y_test": labels[test_idx],
        "X_val": images[val_idx],
        "y_val": labels[val_idx],
    }

## 3) Build/Load Cache and Split Data
This section loads cached CIFAR-10 data (or builds it once), then splits the 10,000 images into train/test/validation sets using configurable percentages.

In [3]:
# Build or load 10,000-image cache, then split into train/test/validation sets.
TRAIN_RATIO = 0.7  # X%
TEST_RATIO = 0.2   # Y%
VAL_RATIO = 0.1    # Z%
SEED = 42

if CACHE_PATH.exists():
    print(f"Loading cached dataset from: {CACHE_PATH}")
    with np.load(CACHE_PATH) as cached:
        images = cached["images"]
        labels = cached["labels"]
        class_names = cached["class_names"].tolist()
else:
    print("Cache not found. Building cache from CIFAR-10...")
    download_and_extract_cifar10()

    # data_batch_1 contains exactly 10,000 training images
    batch_file = EXTRACTED_DIR / "data_batch_1"
    images, labels = load_cifar_batch(batch_file)
    class_names = CLASS_NAMES

    np.savez_compressed(
        CACHE_PATH,
        images=images.astype(np.uint8),
        labels=labels.astype(np.int64),
        class_names=np.array(class_names),
    )
    print(f"Cache saved to: {CACHE_PATH}")

splits = split_dataset(images, labels, TRAIN_RATIO, TEST_RATIO, VAL_RATIO, seed=SEED)

X_train, y_train = splits["X_train"], splits["y_train"]
X_test, y_test = splits["X_test"], splits["y_test"]
X_val, y_val = splits["X_val"], splits["y_val"]

print("Done.")
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")
print(f"unique classes: {len(set(labels.tolist()))}")
print()
print(f"Train set: {X_train.shape[0]} samples ({TRAIN_RATIO:.0%})")
print(f"Test set: {X_test.shape[0]} samples ({TEST_RATIO:.0%})")
print(f"Validation set: {X_val.shape[0]} samples ({VAL_RATIO:.0%})")

Loading cached dataset from: data\cache\cifar10_10000.npz
Done.
images shape: (10000, 32, 32, 3)
labels shape: (10000,)
unique classes: 10

Train set: 7000 samples (70%)
Test set: 2000 samples (20%)
Validation set: 1000 samples (10%)


## 4) Quick Sanity Check
This section verifies split labels and shows a quick peek at the first few labels from each split.

In [4]:
# Validate class labels and inspect first labels from each data split.
print("Train labels (first 10):", y_train[:10].tolist())
print("Test labels (first 10):", y_test[:10].tolist())
print("Validation labels (first 10):", y_val[:10].tolist())
print("First train label name:", class_names[int(y_train[0])])

Train labels (first 10): [0, 0, 9, 2, 7, 8, 9, 3, 0, 1]
Test labels (first 10): [3, 2, 0, 7, 6, 1, 5, 4, 0, 8]
Validation labels (first 10): [0, 9, 7, 0, 0, 0, 3, 8, 7, 6]
First train label name: airplane
